# w9_pfc_5fold.ipynb — 分片 softmax(Partial-FC)五折 @4096/2000ep×5

User (2026-07-23 b):固定分割上 PFC K=8 反超同步、追平全耦合、test_tag .708
高于 I-CE@4096 基准 .698——必须折级硬化拿论文数字。K=8 分片(每片 ~202
游戏 > 窗口 168,168 窗口在片内滑),每步合并 eNow + 8 片窗口成一个精确
softmax,梯度检查点防 OOM(已审计 + 本地实测梯度位级一致)。cv worker 已
移植 --pfc-shards(_pfc8 后缀,cvsel 选点,test_tag readout)。参照 = i2ce
五折(.947/.732/test_tag .698)、swin 五折。判据:PFC 五折 non ≥ 同折 i2ce
且 test_tag ≥ .698 → 大规模方案升论文级正解。AUTO-STOPS。


In [ ]:
# constants
import os

REPO = "/workspace/stable-query-latent"
URL = "https://github.com/Nice9Tian/stable-query-latent.git"
DATA_SRC = "/workspace/fusion_cache_w9"
DATA_RAM = "/dev/shm/fusion_cache_w9"
OUT_DIR = "/workspace/w9_cv_out"

ARM = "wcle_swin168step84loop2i2ce_icetf"
CAP, EPOCHS, N_FOLDS = 4096, 2000, 5
PFC_SHARDS = 8          # K: each shard holds 1/K (~202 games) > SWIN_W=168
os.makedirs(OUT_DIR, exist_ok=True)
print(f"PFC 5-fold: {ARM}@{CAP} {EPOCHS}ep, K={PFC_SHARDS} shards")


In [ ]:
# FORCE-sync repo to origin/main.
import os, importlib.util
if not os.path.isdir(os.path.join(REPO, ".git")):
    !git clone {URL} {REPO}
%cd {REPO}
!git remote set-url origin {URL}
!git fetch origin main
!git reset --hard origin/main
!git rev-parse --short HEAD
for pkg in ("sklearn", "scipy"):
    if importlib.util.find_spec(pkg) is None:
        !pip -q install scikit-learn scipy
        break
import sys
if REPO not in sys.path:
    sys.path.insert(0, REPO)
sys.path.insert(0, os.path.join(REPO, "Pod"))
import w9_jobs as J
print("machinery loaded")


In [ ]:
# Stage the corpus into RAM (same file set as w9_a100.ipynb).
import shutil
from pathlib import Path
REQUIRED = ["games.npz", "wiki_eval.npz", "wscan_gal_rev.npz",
            "wscan_pool_rev.npy", "wscan_pool_rev_rid.npy", "wscan_pool_rev_len.npy",
            "ss_queries_rev.npz", "ss_queries_rev_S.npy",
            "wiki_clean_views.npz", "sp_raw_views.npz", "tag_labels.npz",
            "wiki_eval_split.json", "_tag_splitM.json"]
src = Path(DATA_SRC)
missing = [f for f in REQUIRED if not (src / f).exists()]
assert not missing, f"missing in {DATA_SRC}: {missing}"
dst = Path(DATA_RAM)
dst.mkdir(parents=True, exist_ok=True)
for f in REQUIRED:
    s, d = src / f, dst / f
    if not d.exists() or d.stat().st_size != s.stat().st_size:
        print(f"staging {f} ({s.stat().st_size/1e9:.2f} GB) ...", flush=True)
        shutil.copyfile(s, d)
DATA_DIR = str(dst)
print("corpus in RAM:", DATA_DIR)

In [ ]:
# Full pool: must be READY on the volume; stage onto fast local storage.
import os, time
from pathlib import Path
from Pod.h5_staging import parallel_copy

ready = Path(DATA_SRC) / "full_pool_READY"
assert ready.exists(), "full pool not READY -- run a campaign notebook's build cell once"
src_v = Path(DATA_SRC) / "full_pool_fp16.npy"
src_m = Path(DATA_SRC) / "full_pool_meta.npz"
need = src_v.stat().st_size + (5 << 30)

def _free(p):
    st = os.statvfs(p)
    return st.f_bavail * st.f_frsize

dest_dir = None
for cand in ("/dev/shm", "/root/data", "/root"):
    Path(cand).mkdir(parents=True, exist_ok=True)
    if _free(cand) > need:
        dest_dir = Path(cand)
        break
if dest_dir is None:
    print("WARNING: no local space -- workers will mmap the NETWORK VOLUME copy.")
    FULL_POOL_PATH = str(src_v)
else:
    dst_v = dest_dir / "full_pool_fp16.npy"
    if dst_v.exists() and dst_v.stat().st_size == src_v.stat().st_size:
        print("local full pool already staged:", dst_v)
    else:
        t0 = time.time()
        tmp = dst_v.with_name(dst_v.name + ".copying")
        print(f"staging {src_v.stat().st_size/2**30:.0f} GiB -> {dst_v} ...", flush=True)
        parallel_copy(src_v, tmp, workers=8)
        os.replace(tmp, dst_v)
        print(f"staged in {(time.time()-t0)/60:.1f} min", flush=True)
    import shutil
    shutil.copyfile(src_m, dest_dir / "full_pool_meta.npz")
    FULL_POOL_PATH = str(dst_v)
print("FULL_POOL_PATH =", FULL_POOL_PATH)


In [ ]:
# Run 5 PFC folds, one consumer thread per GPU (folds eaten from a queue).
# A PFC 4096 fold peaks ~30G with the checkpoint fix -> fits a 48G+ card;
# keep one fold per GPU for headroom.
import json, os, subprocess, threading, time
import queue as _q
from pathlib import Path

cdir = Path(OUT_DIR) / "claims"
logd = Path(OUT_DIR) / "logs"
logd.mkdir(parents=True, exist_ok=True)
cdir.mkdir(parents=True, exist_ok=True)
gpus = J.detect_gpus()

todo = []
for k in range(N_FOLDS):
    nm = f"w9cv_{ARM}_fold{k}_g{CAP}_pfc{PFC_SHARDS}"
    if (Path(OUT_DIR) / f"tower_{nm}_fp_ep{EPOCHS}.npz").exists():
        print(f"[skip] {nm} done"); continue
    todo.append((k, nm))
print(f"{len(todo)} fold(s) to run")
stop_evt = threading.Event()
threading.Thread(target=J._monitor, args=([logd], stop_evt), daemon=True).start()
fails = []

def run_one(g, k, nm):
    ok = J.try_claim(cdir, nm)
    if not ok:
        print(f"[claim] {nm} held -- waiting 130s", flush=True)
        time.sleep(130)
        ok = J.try_claim(cdir, nm)
    if not ok:
        print(f"[claim] {nm} held elsewhere -- skipped", flush=True); return
    cmd = ["python", "-u", J.CV_WORKER, "--data-dir", DATA_DIR, "--out-dir",
           OUT_DIR, "--repo", REPO, "--arm", ARM, "--fold", str(k),
           "--n-folds", str(N_FOLDS), "--anchor-cap", str(CAP),
           "--epochs", str(EPOCHS), "--ckpt-every", "50",
           "--pfc-shards", str(PFC_SHARDS),
           "--full-pool", "--full-pool-path", FULL_POOL_PATH,
           "--claim-file", str(cdir / f"{nm}.claim")]
    print(f"[gpu{g}] start {nm}", flush=True)
    t0 = time.time()
    with open(logd / f"{nm}.log", "w") as fh:
        pr = subprocess.run(cmd, stdout=fh, stderr=subprocess.STDOUT,
                            env=dict(os.environ, CUDA_VISIBLE_DEVICES=g,
                                     PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True"))
    if pr.returncode != 0:
        (cdir / f"{nm}.claim").unlink(missing_ok=True); fails.append(nm)
    print(f"[gpu{g}] {'ok' if pr.returncode == 0 else 'FAIL'} {nm} "
          f"[{(time.time() - t0) / 3600:.1f}h]", flush=True)

jq = _q.Queue()
for job in todo:
    jq.put(job)

def gpu_consumer(g, delay):
    time.sleep(delay)
    while True:
        try:
            k, nm = jq.get_nowait()
        except _q.Empty:
            return
        run_one(g, k, nm)

ths = [threading.Thread(target=gpu_consumer, args=(g, 60 * i))
       for i, g in enumerate(gpus)]
for t in ths:
    t.start()
for t in ths:
    t.join()
stop_evt.set()
print(f"done; {len(fails)} failed")


In [ ]:
# Five-fold verdict: PFC K=8 vs i2ce (full coupling) and swin scratch, on
# neu / non / test_tag (the current TAG baseline, testset TAG F1).
import json
import numpy as np
from pathlib import Path

def rows_of(arm, sfx=""):
    out = {}
    for k in range(N_FOLDS):
        p = Path(OUT_DIR) / f"zsbest_w9cv_{arm}_fold{k}_g{CAP}{sfx}_fp.json"
        if p.exists():
            out[k] = json.loads(p.read_text())
    return out

def show(lab, rows):
    if not rows:
        print(f"{lab:26s} (pending)"); return None
    def MS(f):
        v = [r[f] for r in rows.values() if f in r]
        return (np.mean(v), np.std(v)) if v else (float("nan"), 0)
    tt = MS("test_tag")
    print(f"{lab:26s} n={len(rows)} neu {MS('nm_neutral')[0]:.3f}±{MS('nm_neutral')[1]:.3f} "
          f"non {MS('nm_noname')[0]:.3f}±{MS('nm_noname')[1]:.3f} "
          f"test_tag {tt[0]:.3f}±{tt[1]:.3f}")
    return rows

pfc = show(f"PFC K={PFC_SHARDS} (sharded)", rows_of(ARM, f"_pfc{PFC_SHARDS}"))
ic = show("i2ce full coupling", rows_of("wcle_i2ce_icetf"))
sc = show("swin scratch", rows_of(ARM))
# paired deltas vs each reference (same folds)
for ref, lab in ((ic, "vs i2ce"), (sc, "vs swin")):
    if pfc and ref:
        ks = [k for k in pfc if k in ref]
        for f in ("nm_noname", "test_tag"):
            d = [pfc[k][f] - ref[k][f] for k in ks if f in pfc[k] and f in ref[k]]
            if d:
                print(f"  {lab} d{f}: mean {np.mean(d):+.3f} "
                      f"wins {sum(1 for x in d if x>0)}/{len(d)}")


In [ ]:
# AUTO-STOP: stop THIS pod when the queue has finished (results live on the
# network volume; idle GPU time is pure waste). Uses the hardened ladder in
# VICReg_review/pod_selfstop.py. Set AUTO_STOP=False to keep the pod alive.
AUTO_STOP = True
if AUTO_STOP:
    import sys
    if REPO not in sys.path:
        sys.path.insert(0, REPO)
    from VICReg_review import pod_selfstop
    if fails:
        print(f"NOTE: {len(fails)} job(s) FAILED -- logs in {OUT_DIR}/logs; "
              "stopping anyway to avoid idle burn.")
    pod_id, api_key, ctl = pod_selfstop.preflight("")
    pod_selfstop.stop_pod(pod_id, api_key, ctl)
else:
    print("AUTO_STOP disabled -- remember to stop the pod yourself.")